# Лекция 2. Выпуклость: множества, функции, задачи, условие оптимальности

*Вычислительная оптимизация, магистратура, 1 курс. 16 сентября 2026.*

**План лекции**

1. Что мы обещали на лекции 1
2. Выпуклые множества
3. Выпуклые функции
4. Как проверить выпуклость функции
5. Операции, сохраняющие выпуклость
6. Выпуклые задачи и главная теорема
7. Условие оптимальности для выпуклых задач
8. Что это даёт численно
9. Итоги лекции
10. Что дальше
11. Практика на занятии
12. Упражнения

Конспект опирается на разделы 2.4 и главу 3 лекционных заметок М. Диля (*Lecture Notes on Numerical Optimization*, 2017) и главы 2–4 книги Boyd & Vandenberghe. Код демонстраций — ноутбук [`demo02.ipynb`](demo02.ipynb). Картинки конспекта строятся скриптом [`make_figures.py`](make_figures.py).

---

## 1. Что мы обещали на лекции 1

На прошлой лекции мы несколько раз произнесли слово «выпуклость» и каждый раз откладывали точное определение. Картинка из лекции 1: в выпуклой задаче все старты приходят в одну точку, в невыпуклой — каждый в свою.

<img src="../lecture01/img/06_convex_vs_nonconvex.png" width="640" alt="выпуклая и невыпуклая функции: траектории из разных стартов">

Сегодня отвечаем на три вопроса:

1. **Как доказать**, что множество или функция выпуклы, — не «на глаз», а по критерию, который можно проверить руками или кодом.
2. **Почему** для выпуклой задачи любой локальный минимум глобален — главная теорема лекции, и мы её докажем целиком.
3. **Как узнать решение, не запуская метод**: условие оптимальности, которое для выпуклых задач одновременно необходимо и достаточно.

Всё остальное в курсе — методы — будет опираться на эти три ответа. Практический итог к концу пары: вы сможете доказать, что обучение логистической регрессии — задача с единственным ответом, к которому солвер приходит из любого старта (демо, часть d).

## 2. Выпуклые множества

**Определение.** Множество $\Omega \subset \mathbb{R}^n$ **выпукло**, если вместе с любыми двумя точками оно содержит соединяющий их отрезок:

$$
\forall\, x, y \in \Omega,\ t \in [0, 1]: \quad x + t\,(y - x) \in \Omega .
$$

Точку $x + t(y - x) = (1 - t)x + t y$ называют **выпуклой комбинацией** $x$ и $y$. Определение даёт и способ проверки: чтобы доказать *невыпуклость*, достаточно предъявить две точки и вылезающий наружу отрезок; чтобы доказать *выпуклость*, нужно проверить отрезок для всех пар.

<img src="img/01_sets.png" width="640" alt="галерея выпуклых и невыпуклых множеств с тестом-отрезком">

**Примеры выпуклых множеств** (доказательство — подстановка выпуклой комбинации в определение):

| Множество | Запись | Почему выпукло |
|---|---|---|
| аффинное множество (прямая, плоскость) | $\{x : Ax = b\}$ | $A((1-t)x + ty) = (1-t)b + tb = b$ |
| полупространство | $\{x : a^\top x \le b\}$ | $a^\top((1-t)x + ty) = (1-t)a^\top x + t a^\top y \le b$ |
| многогранник | $\{x : Ax \le b\}$ | пересечение полупространств (см. ниже) |
| шар | $\{x : \Vert x - c\Vert \le r\}$ | неравенство треугольника |
| эллипсоид | $\{x : (x-c)^\top Q (x - c) \le 1\}$, $Q \succ 0$ | аффинный образ шара (см. ниже) |
| надграфик выпуклой функции | раздел 3 | по определению выпуклой функции |

**Примеры невыпуклых:** окружность $\{\Vert x\Vert = r\}$ (хорда идёт внутри круга), кольцо, объединение двух непересекающихся шаров, любое конечное множество из двух и более точек, целочисленная решётка $\mathbb{Z}^n$.

**Операции, сохраняющие выпуклость множеств.**

1. **Пересечение** любого (даже бесконечного) числа выпуклых множеств выпукло: отрезок лежит в каждом из них, значит, и в пересечении.
2. **Аффинный образ**: если $\Omega$ выпукло, то $\{Ax + b : x \in \Omega\}$ выпукло.
3. **Аффинный прообраз**: если $\Omega$ выпукло, то $\{z : Az + b \in \Omega\}$ выпукло.

Пересечение — главный инструмент: многогранник $\{Ax \le b\}$ выпукл, потому что это пересечение полупространств $a_i^\top x \le b_i$. Именно так устроено допустимое множество LP.

<img src="img/02_intersection.png" width="440" alt="многогранник как пересечение полуплоскостей">

Объединение выпуклых множеств, как правило, **не** выпукло (два шара на картинке выше).

<details>
<summary>Ещё один важный выпуклый конус (для справки)</summary>

Множество симметричных положительно полуопределённых матриц $\mathcal{S}^n_+ = \{X = X^\top : X \succeq 0\}$ выпукло: если $v^\top X v \ge 0$ и $v^\top Y v \ge 0$ для всех $v$, то и $v^\top((1-t)X + tY)v \ge 0$. Его аффинные прообразы $\{x : A_0 + \sum_i x_i A_i \succeq 0\}$ называются **линейными матричными неравенствами** (LMI); на них построено полуопределённое программирование (SDP), в которое вкладываются LP, QP и QCQP. В курсе SDP не рассматривается.

</details>

## 3. Выпуклые функции

**Определение.** Функция $f: \Omega \to \mathbb{R}$ на выпуклом множестве $\Omega$ **выпукла**, если любая хорда её графика лежит не ниже графика:

$$
\forall\, x, y \in \Omega,\ t \in [0, 1]: \quad f\big((1-t)x + ty\big) \le (1-t) f(x) + t f(y).
$$

Если при $x \ne y$ и $t \in (0, 1)$ неравенство строгое, функция **строго выпукла**. Функция **вогнута**, если $-f$ выпукла. Аффинная функция $a^\top x + b$ одновременно выпукла и вогнута — единственный такой случай.

<img src="img/03_secant_epigraph.png" width="720" alt="хорда над графиком, хорда под графиком, надграфик">

Определение можно переформулировать через множество: **надграфик** (epigraph)

$$
\operatorname{epi} f = \{(x, s) \in \mathbb{R}^n \times \mathbb{R} : x \in \Omega,\ s \ge f(x)\}
$$

выпукл тогда и только тогда, когда $f$ выпукла. Так выпуклость функций сводится к выпуклости множеств, и все результаты раздела 2 переносятся на функции.

**Примеры выпуклых функций.**

| Функция | Область | Выпукла, потому что |
|---|---|---|
| $a^\top x + b$ | $\mathbb{R}^n$ | равенство в определении |
| $e^{x}$, $x^2$, $x^4$ | $\mathbb{R}$ | вторая производная неотрицательна (раздел 4) |
| $\lvert x \rvert$ | $\mathbb{R}$ | по определению: негладкая, второй производной в нуле нет |
| $-\log x$, $1/x$ | $x > 0$ | вторая производная положительна |
| любая норма $\Vert x\Vert$ | $\mathbb{R}^n$ | неравенство треугольника плюс однородность |
| $\tfrac12 x^\top Q x + c^\top x$ при $Q \succeq 0$ | $\mathbb{R}^n$ | гессиан $Q \succeq 0$ (раздел 4) |
| $\max(x_1, \dots, x_n)$ | $\mathbb{R}^n$ | максимум аффинных (раздел 5) |
| $\log \sum_i e^{x_i}$ | $\mathbb{R}^n$ | гессиан $\succeq 0$ (проверка — упражнение 12.2) |

**Не выпуклы:** $x^3$, $\sin x$, $\sqrt{x}$ (вогнута), $(x^2 - 1)^2 + 0.3x$ из лекции 1 — у всех найдётся хорда под графиком.

## 4. Как проверить выпуклость функции

Проверять определение для всех пар точек неудобно. Для гладких функций есть два критерия.

**Теорема (критерий первого порядка).** Пусть $f \in C^1$ на выпуклом открытом $\Omega$. Тогда $f$ выпукла тогда и только тогда, когда её график лежит не ниже любой касательной:

$$
\forall\, x, y \in \Omega: \quad f(y) \ge f(x) + \nabla f(x)^\top (y - x).
$$

<img src="img/04_tangent.png" width="640" alt="касательные под графиком выпуклой функции">

Это неравенство — рабочая лошадка всего курса: линейная модель $f(x) + \nabla f(x)^\top(y - x)$ является **глобальной нижней оценкой** для $f$. Именно поэтому для выпуклых функций из локальной информации (градиента в одной точке) следуют глобальные выводы — в разделе 7 мы получим из него условие оптимальности.

**Теорема (критерий второго порядка).** Пусть $f \in C^2$ на выпуклом открытом $\Omega$. Тогда $f$ выпукла тогда и только тогда, когда гессиан положительно полуопределён всюду:

$$
\forall\, x \in \Omega: \quad \nabla^2 f(x) \succeq 0 .
$$

Если $\nabla^2 f(x) \succ 0$ всюду, $f$ строго выпукла (обратное неверно: $x^4$ строго выпукла, но $f''(0) = 0$).

<details>
<summary>Доказательства обоих критериев</summary>

*Критерий первого порядка.* «$\Rightarrow$»: из определения $f(x + t(y-x)) - f(x) \le t\,(f(y) - f(x))$; делим на $t$ и устремляем $t \to 0$: слева получаем $\nabla f(x)^\top(y - x)$. «$\Leftarrow$»: для $z = (1-t)x + ty$ запишем неравенство касательной в точке $z$ дважды, для $x$ и для $y$: $f(x) \ge f(z) + \nabla f(z)^\top(x - z)$, $f(y) \ge f(z) + \nabla f(z)^\top(y - z)$. Умножим на $(1-t)$ и $t$ и сложим: $(1-t)f(x) + tf(y) \ge f(z) + \nabla f(z)^\top\big[(1-t)x + ty - z\big] = f(z)$.

*Критерий второго порядка.* «$\Rightarrow$»: по Тейлору $f(x + tp) = f(x) + t\nabla f(x)^\top p + \tfrac12 t^2 p^\top \nabla^2 f(x) p + o(t^2)$; левая часть минус первые два слагаемых неотрицательна по критерию первого порядка, делим на $t^2/2$ и устремляем $t \to 0$: $p^\top \nabla^2 f(x) p \ge 0$. «$\Leftarrow$»: формула Тейлора с остаточным членом в форме Лагранжа, $f(y) = f(x) + \nabla f(x)^\top(y-x) + \tfrac12 (y-x)^\top \nabla^2 f(\xi)(y - x)$ с $\xi$ на отрезке $[x, y]$; последнее слагаемое $\ge 0$, получаем неравенство касательной. $\square$

</details>

**Примеры.**

- Квадратичная $f(x) = \tfrac12 x^\top Q x + c^\top x$: гессиан $\nabla^2 f = Q$ не зависит от $x$, поэтому $f$ выпукла $\iff Q \succeq 0$, строго выпукла $\iff Q \succ 0$. Практическая проверка — `np.linalg.eigvalsh(Q).min() >= -1e-10`: допуск нужен, потому что нулевое собственное число полуопределённой $Q$ численно выходит как $\pm 10^{-16}$.
- Линейный МНК $\tfrac12\Vert Ax - b\Vert^2$: $Q = A^\top A$, и $v^\top A^\top A v = \Vert Av\Vert^2 \ge 0$ — выпукла всегда, строго выпукла при полном столбцовом ранге $A$.
- $e^x$: $f'' = e^x > 0$. $-\log x$ при $x > 0$: $f'' = 1/x^2 > 0$.
- $(x_1^2 - 1)^2 + x_2^2$: гессиан $\operatorname{diag}(12x_1^2 - 4,\ 2)$, при $|x_1| < 1/\sqrt{3}$ первое собственное число отрицательно — не выпукла.

<img src="img/05_hessian_map.png" width="720" alt="карта наименьшего собственного числа гессиана">

Карта $\lambda_{\min}\nabla^2 f(x)$ — то, что можно посчитать численно для любой функции двух переменных (демо, часть a). Красная полоса — область, где гессиан индефинитен; одной такой точки достаточно, чтобы функция не была выпуклой.

<details>
<summary>Пример посложнее: $f(x, t) = x^\top x / t$ выпукла при $t > 0$</summary>

Гессиан $\nabla^2 f = \dfrac{2}{t^3}\begin{pmatrix} t^2 I & -t x \\ -t x^\top & x^\top x\end{pmatrix}$, и для $v = (z, s)$ имеем $v^\top \nabla^2 f\, v = \dfrac{2}{t^3}\Vert t z - s x\Vert^2 \ge 0$. Такие функции («перспектива» квадрата нормы) появляются в робастной оптимизации и статистике.

</details>

## 5. Операции, сохраняющие выпуклость

Критерий второго порядка требует гессиана; часто проще собрать функцию из выпуклых «кирпичей» операциями, которые выпуклость сохраняют (демо, часть b).

1. **Неотрицательная взвешенная сумма.** $\alpha f + \beta g$ выпукла при $\alpha, \beta \ge 0$ и выпуклых $f$, $g$. То же для интегралов и бесконечных сумм.
2. **Композиция с аффинным отображением.** Если $f$ выпукла, то $\tilde f(x) = f(Ax + b)$ выпукла. Так $\Vert Ax - b\Vert$ выпукла для любой нормы.
3. **Поточечный максимум и супремум.** $f(x) = \max_i f_i(x)$ и $f(x) = \sup_{i \in I} f_i(x)$ выпуклы, если выпуклы все $f_i$. Надграфик максимума — пересечение надграфиков.
4. **Композиция с монотонной выпуклой функцией.** Если $f$ выпукла, а $g: \mathbb{R} \to \mathbb{R}$ выпукла и не убывает, то $g \circ f$ выпукла. Например, $e^{f(x)}$ и $f(x)^2$ при $f \ge 0$. Условие монотонности существенно: $g(u) = -u$ выпукла (аффинна), но $-f$ не выпукла.

<img src="img/06_max_affine.png" width="480" alt="максимум аффинных функций">

<details>
<summary>Почему композиция с монотонной выпуклой сохраняет выпуклость</summary>

Для $C^2$ функций: $\nabla^2 (g \circ f)(x) = g''(f(x))\,\nabla f(x) \nabla f(x)^\top + g'(f(x))\,\nabla^2 f(x)$. Первое слагаемое $\succeq 0$, так как $g'' \ge 0$ и $\nabla f \nabla f^\top \succeq 0$; второе $\succeq 0$, так как $g' \ge 0$ и $\nabla^2 f \succeq 0$. В общем случае — прямо из определений.

</details>

**Мост к множествам: подуровневые множества.** Если $f$ выпукла, то для любого $c$ множество $\{x : f(x) \le c\}$ выпукло: из $f(x) \le c$, $f(y) \le c$ следует $f((1-t)x + ty) \le (1-t)c + tc = c$. Обратное неверно ($\sqrt{|x|}$ имеет выпуклые подуровневые множества, но не выпукла).

<img src="img/07_sublevel.png" width="720" alt="подуровневые множества выпуклой и невыпуклой функции">

Отсюда главное для нас следствие: ограничение $h_i(x) \ge 0$ с **вогнутой** $h_i$ задаёт выпуклое множество (это подуровневое множество выпуклой $-h_i$), а пересечение таких множеств выпукло.

**Примеры сборки.**

- $\Vert Ax - b\Vert_2^2 + \lambda \Vert x\Vert_1$ (LASSO): первое слагаемое — квадратичная с $Q = 2A^\top A \succeq 0$, второе — норма; сумма с $\lambda \ge 0$ выпукла.
- $\max_i \lvert a_i^\top x - b_i\rvert$ (чебышёвская подгонка, бонус ДЗ 1): каждый модуль — максимум двух аффинных функций, максимум максимумов выпукл.
- Логистическая потеря $\sum_i \log\big(1 + e^{-y_i a_i^\top x}\big)$: $\log(1 + e^{u})$ выпукла по $u$ (вторая производная $e^u/(1+e^u)^2 > 0$), композиция с аффинной $u = -y_i a_i^\top x$ и сумма. Обучение логистической регрессии — выпуклая задача.
- $(x^2 - 1)^2$: композиция выпуклой $u^2$ с выпуклой $u = x^2 - 1$, но $u^2$ **не монотонна** — правило не применяется, и функция действительно не выпукла.

## 6. Выпуклые задачи и главная теорема

**Определение.** Задача $\min_{x \in \Omega} f(x)$ **выпукла**, если $\Omega$ — выпуклое множество и $f$ — выпуклая функция на $\Omega$.

В стандартной форме (NLP) из лекции 1 достаточное условие такое.

**Теорема (стандартная форма выпуклой NLP).** Если $f$ выпукла, все $g_i$ аффинны, а все $h_i$ вогнуты, то задача

$$
\min_x f(x) \quad\text{при}\quad g(x) = 0,\ h(x) \ge 0
$$

выпукла.

*Доказательство.* Множество $\{g(x) = 0\}$ аффинно, множества $\{h_i(x) \ge 0\}$ выпуклы как подуровневые множества выпуклых $-h_i$; $\Omega$ — их пересечение. $\square$

Равенства обязаны быть аффинными: нелинейное равенство $g(x) = 0$ задаёт «кривую», и отрезок между её точками с неё сходит (окружность из упражнения 12.1б лекции 1). Неравенство же может быть нелинейным, лишь бы $h_i$ была вогнутой: $h(x) = 1 - \Vert x\Vert^2$ вогнута, и шар выпукл.

> **Замечание о соглашениях.** В книгах по выпуклой оптимизации (Boyd & Vandenberghe) неравенства пишут как $f_i(x) \le 0$ с выпуклыми $f_i$, а равенства сразу как $Ax = b$. Это та же самая форма: $f_i = -h_i$.

**Теорема (локальный минимум выпуклой задачи глобален).** Если задача выпукла, то любой её локальный минимум является глобальным.

*Доказательство.* Пусть $x^\ast$ — локальный минимум, $y \in \Omega$ — любая допустимая точка; покажем $f(y) \ge f(x^\ast)$. По локальной оптимальности есть окрестность $\mathcal{N}$ точки $x^\ast$, в которой $f(\tilde x) \ge f(x^\ast)$ для всех $\tilde x \in \Omega \cap \mathcal{N}$. Отрезок от $x^\ast$ до $y$ целиком лежит в $\Omega$ (множество выпукло). Возьмём на нём точку $\tilde x = x^\ast + t(y - x^\ast)$ с настолько малым $t \in (0, 1]$, что $\tilde x \in \mathcal{N}$. Тогда

$$
f(x^\ast) \le f(\tilde x) \le (1 - t) f(x^\ast) + t f(y),
$$

где первое неравенство — локальная оптимальность, второе — выпуклость $f$. Отсюда $t\,(f(y) - f(x^\ast)) \ge 0$ и, так как $t > 0$, $f(y) \ge f(x^\ast)$. $\square$

<img src="img/08_local_global_proof.png" width="460" alt="схема доказательства: отрезок от x* к y проходит через окрестность">

Доказательство использовало ровно два факта: отрезок лежит в $\Omega$ (выпуклость множества) и $f$ на отрезке не выше хорды (выпуклость функции). Уберите любое — и появятся локальные минимумы, которые не глобальны (лекция 1, пример 7.4).

**Следствия.**

- Множество решений выпуклой задачи выпукло (оно есть подуровневое множество $\{f \le f^\ast\} \cap \Omega$). Решений может быть много, но они образуют «одну яму», а не несколько.
- Если $f$ **строго** выпукла, решение единственно: из двух решений $x^\ast \ne y^\ast$ середина отрезка дала бы значение строго меньше $f^\ast$.
- Выпуклость ничего не говорит о *существовании* решения: $\min_x e^x$ — выпуклая задача без минимума. За существование по-прежнему отвечают теорема Вейерштрасса и коэрцитивность (лекция 1, раздел 5); выпуклость гарантирует лишь, что найденный локальный минимум — глобальный.

**Классы выпуклых задач.** LP (всё аффинно) $\subset$ выпуклая QP ($Q \succeq 0$, ограничения аффинные) $\subset$ выпуклая QCQP (цель и все $-h_i$ — выпуклые квадратичные). Все они — частные случаи выпуклой NLP, и для каждого есть специализированные солверы, которые мы будем изучать в части IV.

**Чек-лист: как распознать выпуклую задачу.** Расширяет список из лекции 1 (раздел 6.3) и достаточен для ДЗ 2.

1. Цель: аффинна, норма, квадратичная с $Q \succeq 0$, $e^{(\cdot)}$, $-\log$, $\log\sum e$, максимум выпуклых, композиция с аффинным — выпукла. Проверка гессиана по критерию второго порядка, если функция гладкая и «нестандартная».
2. Равенства — только аффинные.
3. Неравенства $h_i(x) \ge 0$ — с вогнутыми $h_i$: аффинные, $r^2 - \Vert x - c\Vert^2$, $\log x_i$. Не проходят проверку $\Vert x\Vert^2 - r^2 \ge 0$ (внешность шара — множество и правда не выпукло) и $x_1 x_2 \ge 1$: здесь $h = x_1 x_2 - 1$ не вогнута, без условия $x > 0$ множество — две ветви гиперболы, не выпукло, а вот $\{x_1 x_2 \ge 1,\ x > 0\}$ выпукло (упражнение 12.1в). Чек-лист — достаточное условие, а не критерий: смотреть нужно на множество, а не только на формулу $h$.
4. Если и цель, и множество прошли проверку — задача выпукла, и любой найденный локальный минимум глобален. Если хоть что-то не прошло — задача *может* быть невыпуклой, и нужно либо доказать невыпуклость (найти хорду / индефинитный гессиан), либо переформулировать.

| Задача из лекции 1 | Выпукла? | Почему |
|---|---|---|
| линейный МНК (7.1) | да | квадратичная, $A^\top A \succeq 0$, без ограничений |
| планирование производства (7.2) | да | LP |
| цепь с полом (7.3) | да | квадратичная энергия с $Q \succ 0$, линейные неравенства |
| $(x^2 - 1)^2 + 0.3x$ (7.4) | нет | $f'' = 12x^2 - 4 < 0$ при $\lvert x\rvert < 1/\sqrt{3}$ |
| ближайшая точка окружности (упр. 12.1б) | нет | нелинейное равенство |
| нелинейный МНК $x_1 e^{-x_2 t}$ (упр. 12.1г) | нет | гессиан индефинитен на плато |

## 7. Условие оптимальности для выпуклых задач

Для выпуклой задачи можно сказать, является ли точка решением, глядя только на градиент в ней и на форму $\Omega$.

**Теорема.** Пусть $\Omega$ выпукло, а $f$ выпукла и непрерывно дифференцируема на открытом множестве, содержащем $\Omega$ (тогда критерий первого порядка применим и в граничных точках $\Omega$). Точка $x^\ast \in \Omega$ — глобальное решение задачи $\min_{x \in \Omega} f(x)$ тогда и только тогда, когда

$$
\nabla f(x^\ast)^\top (y - x^\ast) \ge 0 \quad \text{для всех } y \in \Omega .
$$

*Доказательство.* «$\Leftarrow$»: по критерию первого порядка $f(y) \ge f(x^\ast) + \nabla f(x^\ast)^\top(y - x^\ast) \ge f(x^\ast)$ для любого $y \in \Omega$. «$\Rightarrow$»: если для некоторого $y \in \Omega$ выполнено $\nabla f(x^\ast)^\top (y - x^\ast) < 0$, то по Тейлору $f(x^\ast + t(y - x^\ast)) = f(x^\ast) + t\,\nabla f(x^\ast)^\top(y - x^\ast) + o(t) < f(x^\ast)$ при малых $t > 0$, а точка $x^\ast + t(y - x^\ast)$ допустима по выпуклости $\Omega$ — противоречие с оптимальностью. $\square$

<img src="img/09_optimality.png" width="720" alt="условие оптимальности: антиградиент смотрит наружу">

**Геометрия.** Условие говорит: из $x^\ast$ нет ни одного допустимого направления $y - x^\ast$, вдоль которого $f$ убывает в первом порядке. Антиградиент $-\nabla f(x^\ast)$ образует тупой (или прямой) угол со всеми направлениями внутрь $\Omega$ — он «смотрит наружу», и ограничения не дают сделать шаг. Если $x^\ast$ лежит внутри $\Omega$, направления $y - x^\ast$ есть во все стороны, и условие превращается в $\nabla f(x^\ast) = 0$.

**Следствие (безусловная выпуклая задача).** Если $f$ выпукла и $\Omega = \mathbb{R}^n$, то $x^\ast$ — глобальный минимум тогда и только тогда, когда $\nabla f(x^\ast) = 0$.

Для невыпуклых задач $\nabla f = 0$ — лишь необходимое условие (стационарная точка может быть максимумом или седлом); об этом лекция 4.

**Пример 1: квадратичная функция.** Для $f(x) = \tfrac12 x^\top Q x + c^\top x$ с $Q \succ 0$ условие $\nabla f = Qx + c = 0$ даёт единственное решение $x^\ast = -Q^{-1} c$, а оптимальное значение

$$
\min_x \Big(\tfrac12 x^\top Q x + c^\top x\Big) = -\tfrac12\, c^\top Q^{-1} c .
$$

Эта формула будет постоянно появляться в части II: метод Ньютона на каждом шаге решает именно такую задачу, только на месте $Q$ стоит гессиан (или его приближение).

**Пример 2: проекция на выпуклое множество.** Задача $\min_{x \in \Omega} \tfrac12 \Vert x - p\Vert^2$ — найти ближайшую к $p$ точку выпуклого $\Omega$. Цель строго выпукла, решение единственно и называется **проекцией** $P_\Omega(p)$. Условие оптимальности с $\nabla f(x^\ast) = x^\ast - p$:

$$
(x^\ast - p)^\top (y - x^\ast) \ge 0 \quad \forall\, y \in \Omega ,
$$

то есть угол между $p - x^\ast$ и любым направлением внутрь $\Omega$ не острый. Два случая, где ответ выписывается явно:

- шар $\Vert x\Vert \le 1$: при $\Vert p\Vert > 1$ проекция $x^\ast = p / \Vert p\Vert$ (упражнение 12.1б лекции 1 было ровно об этом, только с равенством);
- параллелепипед $l \le x \le u$: проекция покоординатная, $x^\ast_i = \min(\max(p_i, l_i), u_i)$ — операция `np.clip` (упражнение 12.4).

<img src="img/10_projection.png" width="720" alt="проекция на шар и на параллелепипед">

Проекции — строительный блок методов для задач с простыми ограничениями (проекция градиента) и всех методов внутренней точки; мы вернёмся к ним в части IV.

## 8. Что это даёт численно

**Старт не важен — важна только скорость.** Для выпуклой задачи любой метод, сходящийся к точке, где выполнено условие оптимальности, находит глобальное решение. Из какого $x_0$ начинать — вопрос числа итераций, а не правильности ответа (демо, часть d: логистическая регрессия из 10 стартов приходит в одну точку с точностью до критерия остановки).

**Критерий остановки имеет смысл.** Для выпуклой безусловной задачи из $\Vert\nabla f(x_k)\Vert \le \varepsilon$ следует, что $x_k$ близка к глобальному минимуму (насколько близка — зависит от обусловленности, лекция 4). Для невыпуклой из того же неравенства следует только близость к какой-то стационарной точке.

**Сертификат решения.** Условие раздела 7 можно *проверить* для найденной точки, не зная истинного решения: для задач с простыми $\Omega$ — прямо по формуле (демо, часть c), для общих — через двойственность, которая даёт нижнюю границу $f^\ast$ (лекция 3). Если найдено $x$ с $f(x)$, равным нижней границе, решение доказано.

**Моделирование ради выпуклости.** Приём со вспомогательными переменными из лекции 1 (модуль → два линейных неравенства) — это способ записать выпуклую негладкую задачу как выпуклую гладкую. Общее правило: если задачу удаётся сформулировать выпукло, её удастся решить надёжно; невыпуклая формулировка той же задачи может оказаться безнадёжной. Два классических примера. Квадрат нормы вместо нормы: $\tfrac12\Vert r(x)\Vert^2$ вместо $\Vert r(x)\Vert$ — тот же минимум, но гладкая цель, и выпуклость при этом не теряется. Сторона неравенства: $\Vert x\Vert^2 \le 1$ — шар, выпуклое множество; $\Vert x\Vert^2 \ge 1$ — та же функция, но внешность шара, и выпуклости больше нет.

**Границы применимости.** Выпуклость не означает «легко»: выпуклая, но негладкая задача с миллионом переменных может быть трудна, и для неё нужны свои методы. Но невыпуклость почти всегда означает «трудно» — в лучшем случае мы найдём локальный минимум и не сможем доказать, что он глобальный.

## 9. Итоги лекции

1. Множество выпукло, если содержит отрезок между любыми двумя своими точками; функция выпукла, если хорда лежит над графиком (равносильно: надграфик выпукл).
2. Для гладких функций выпуклость проверяется критериями: касательная под графиком ($C^1$) или $\nabla^2 f \succeq 0$ всюду ($C^2$). Для квадратичных функций — собственные числа $Q$.
3. Выпуклость сохраняют: пересечение множеств, аффинные образы и прообразы; неотрицательные суммы, композиция с аффинным отображением, максимум, композиция с монотонной выпуклой функцией. Подуровневые множества выпуклой функции выпуклы.
4. Задача выпукла, если $f$ выпукла, равенства аффинны, а $h_i$ вогнуты. Тогда любой локальный минимум глобален, множество решений выпукло, а при строгой выпуклости решение единственно.
5. Для выпуклой задачи $x^\ast$ — решение тогда и только тогда, когда $\nabla f(x^\ast)^\top(y - x^\ast) \ge 0$ для всех допустимых $y$; без ограничений — $\nabla f(x^\ast) = 0$.
6. Численно: для выпуклых задач старт влияет только на число итераций, критерий остановки гарантирует близость к глобальному решению, а решение можно сертифицировать.

## 10. Что дальше

**Лекция 3 — как учитывать ограничения.** Условие раздела 7 работает, когда допустимые направления видны глазами; в общем случае мы посмотрим на картинку: где живёт минимум, что происходит у стенки и в вершине, почему у каждого ограничения появляется множитель Лагранжа — сила, с которой стенка держит точку, и одновременно цена ослабления ограничения. Функция Лагранжа $L(x,\lambda,\mu)=f(x)-\lambda^\top g(x)-\mu^\top h(x)$ появится как бухгалтерия этих сил; двойственность — второй способ находить те же множители — отложена до части IV.

**Домашнее задание 2** — [`homeworks/hw02.md`](../../homeworks/hw02.md): выпуклость множеств, функций и задач; проекция и условие оптимальности; выпуклая подгонка против невыпуклой. Выдаётся 23 сентября, срок — 7 октября.

**Литература к лекции.** Diehl, разделы 2.4 и 3; Boyd & Vandenberghe, главы 2 (множества), 3 (функции), 4.1–4.2 (выпуклые задачи и условие оптимальности); Поляк, гл. 1, § 1.1–1.2; Нестеров, гл. 2.

## 11. Практика на занятии

Вторая половина пары, около 40 минут. Всё, что нужно, — ноутбук [`demo02.ipynb`](demo02.ipynb).

| Время | Что делаем |
|-------|-----------|
| 10 мин | `demo02.ipynb`, часть (a) вживую: тест хорды на случайных парах и карта $\lambda_{\min}$ гессиана — на что похожа проверка выпуклости «кодом», и почему не найти контрпример не значит доказать. |
| 10 мин | Упражнение 12.1 у доски: пять множеств, по одному на студента — выпукло или нет, с доказательством или отрезком-контрпримером. |
| 10 мин | Часть (c) ноутбука: QP с параллелепипедом, активные границы, проверка условия оптимальности на случайных $y$; упражнение 12.4 — вывести `clip` из условия. |
| 10 мин | Часть (d): логистическая регрессия из 10 стартов против нелинейного МНК из тех же стартов; обсуждение 12.5. |

## 12. Упражнения

Для разбора на занятии и самостоятельно. Подробный разбор с проверкой кодом — ноутбук [`exercises02.ipynb`](exercises02.ipynb), краткие ответы — [`exercises02.md`](exercises02.md).

**12.1.** Выпуклы ли множества? Докажите или предъявите две точки и вылезающий отрезок:

(а) $\{x \in \mathbb{R}^n : \Vert x - a\Vert_2 \le r\}$;

(б) $\{x \in \mathbb{R}^n : x^\top Q x \le 1\}$ при $Q \succ 0$; а при $Q$ с отрицательным собственным числом?

(в) $\{x \in \mathbb{R}^2 : x_1 x_2 \ge 1,\ x_1 > 0,\ x_2 > 0\}$;

(г) $\{x \in \mathbb{R}^2 : x_1^2 + x_2^2 = 1\}$;

(д) объединение двух шаров $\{\Vert x\Vert \le 1\} \cup \{\Vert x - (3, 0)\Vert \le 1\}$.

**12.2.** Выпуклы ли функции (используйте критерий второго порядка или операции из раздела 5)?

(а) $f(x_1, x_2) = x_1^2 / x_2$ на $x_2 > 0$;

(б) $f(x) = -\log x$ на $x > 0$ и $f(x) = x \log x$ на $x > 0$;

(в) $f(x) = x^4 - x^2$;

(г) $f(x) = \log\big(e^{x_1} + e^{x_2}\big)$.

**12.3.** С помощью операций раздела 5 докажите выпуклость (а) $\Vert Ax - b\Vert_2^2 + \lambda\Vert x\Vert_1$, $\lambda \ge 0$; (б) $\max_i \lvert a_i^\top x - b_i\rvert$; (в) $f(x) = \sum_i \log(1 + e^{-y_i a_i^\top x})$. Какие из них гладкие?

**12.4.** Проекция на параллелепипед: решите $\min_x \tfrac12\Vert x - p\Vert^2$ при $l \le x \le u$. Из условия оптимальности раздела 7 выведите, что $x^\ast_i = \min(\max(p_i, l_i), u_i)$. Проверьте численно через `scipy.optimize.minimize` с `bounds`.

**12.5.** (а) Докажите, что множество решений выпуклой задачи выпукло, а при строго выпуклой $f$ решение единственно. (б) Приведите выпуклую задачу с бесконечным множеством решений. (в) Объясните, почему в примере 7.4 лекции 1 старт влияет на ответ, а в логистической регрессии (демо, часть d) — нет.

**Домашнее задание 2** — [`homeworks/hw02.ipynb`](../../homeworks/hw02.ipynb), выдаётся 23 сентября, срок сдачи 7 октября.

## Вопросы для самопроверки

1. Дайте определения выпуклого множества и выпуклой функции. Как они связаны через надграфик?
2. Сформулируйте критерии выпуклости первого и второго порядка. Как проверить выпуклость квадратичной функции?
3. Какой должна быть стандартная форма (NLP), чтобы задача была выпуклой? Почему равенства могут быть только аффинными?
4. Сформулируйте условие оптимальности для выпуклой задачи и объясните его геометрический смысл. Во что оно превращается без ограничений?